# Analysis: Experiment \#A001 - Checking pdf metadata

**Technical summary**
| Exp Nr | Responsible | Date | Description |
| :----: | :---------- | :--: | :---------- |
| A001 | Mario Tormo Romero | 2026-03-03 | Analyzing metadata in pdf files |

<br>

| Results |
| :------ |
| Short text with the results of the experiment |

---
## Objective
In this experiment we aim to determine the amount and quality of the metadata contained in pdf files in different time periods, and thus, with different formats.

---
## Setting
In this experiment we use the following libraries/tools:
- [PyPDF2](https://pypdf2.readthedocs.io/) for extracting the metadata from the pdf files
- [pandas](https://pandas.pydata.org/) for analyzing the metadata

---
## Experiment evaluation
Here the results

---
## Conclusion
What have we learnt from this experiment

---
## Links
Here a list with links to objects, logs and reports

---
## Dev

### 1. Setting up
Preparing libraries and constants.

In [1]:
from pathlib import Path
from PyPDF2 import PdfReader
import re

import pandas as pd

from datetime import datetime


DATA_PATH = Path('../03_data/')

### 2. Extracting metadata from a file
We define now a function for extracting the metadata from a specific file.

In [2]:
def extract_pdf_metadata(pdf_path: Path) -> dict:
    """Extract metadata from a single PDF file."""
    try:
        reader = PdfReader(pdf_path)
        metadata = reader.metadata or {}

        # Convert metadata keys to readable strings
        clean_metadata = {
            key.lstrip("/"): value
            for key, value in metadata.items()
        }

        return clean_metadata

    except Exception as e:
        return {}

# Test for the function
file_path = DATA_PATH / "Ausarbeitungen/WD 1-019-24; WD 7-060-24.pdf"
metadata_dict = extract_pdf_metadata(file_path)
metadata_dict

{'Subject': '',
 'Author': 'Wissenschaftliche Dienste des Deutschen Bundestages',
 'Keywords': '',
 'CreationDate': "D:20241017112032+02'00'",
 'ModDate': "D:20241017112032+02'00'",
 'Title': 'Sachstand - Die Lootbox im deutschen Recht und im europäischen Vergleich'}

In [ ]:


# Test for the function
file_path = DATA_PATH / "Extra" / "ab_2023/Ausarbeitungen/WD 5/WD 5-148-07.pdf"
metadata_dict = extract_pdf_metadata(file_path)
metadata_dict

{}

### 3. Converting dates to datetime format
We use a function for turning strings with dates into datetime objects.

In [3]:
def convert_date(date_str: str) -> datetime | None:
    """Convert a string containing pdf date info into a datetime object."""
    expected_length = 23

    if date_str:
        match = re.match(
            r"(D:\d{4})(\d{2})?(\d{2})?(\d{2})?(\d{2})?(\d{2})?([+\-Z])?(\d{2})?'?(\d{2})?'?",
            date_str,
        )

        if not (date_str.startswith("D:") and match and len(date_str) == expected_length):
            return date_str
    else:
        return date_str

    try:
        date_str = date_str[2:].replace(
            "'", ""
        )  # Remove the "D:" prefix and the single quotation marks
        dt = datetime.strptime(
            date_str, "%Y%m%d%H%M%S%z"
        )  # Turn the date into datetime object

        return dt
    except Exception:
        return date_str

# Test for the function
pdf_date_str = "D:20241017112031+02'00'"
dt_obj = convert_date(pdf_date_str)
print(dt_obj)

2024-10-17 11:20:31+02:00


In [4]:
pdf_date_str = None
dt_obj = convert_date(pdf_date_str)
print(dt_obj)

None


### 4. Updating our original function
We add a call to the new function in the original <code>extract_pdf_metadata</code> function. We will add custom exceptions.

In [5]:
def extract_pdf_metadata(pdf_path: Path) -> dict:
    """
    Summary line.

    Extract metadata from a single PDF file and clean it by
    stripping slashes from the keys and turning strings
    containing dates into datetime objects.
    """
    date_keys = ["CreationDate", "ModDate"]

    try:
        reader = PdfReader(pdf_path)
        metadata = reader.metadata or {}

        # Convert metadata keys to readable strings
        clean_metadata = {key.lstrip("/"): value for key, value in metadata.items()}

        for key in date_keys:
            clean_metadata[key] = convert_date(clean_metadata[key])

        return clean_metadata

    except Exception:
        return {}
    
# Test for the function
file_path = DATA_PATH / "Ausarbeitungen/WD 1-019-24; WD 7-060-24.pdf"
metadata_dict = extract_pdf_metadata(file_path)
metadata_dict

{'Subject': '',
 'Author': 'Wissenschaftliche Dienste des Deutschen Bundestages',
 'Keywords': '',
 'CreationDate': datetime.datetime(2024, 10, 17, 11, 20, 32, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))),
 'ModDate': datetime.datetime(2024, 10, 17, 11, 20, 32, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))),
 'Title': 'Sachstand - Die Lootbox im deutschen Recht und im europäischen Vergleich'}

### 5. Parsing all pdfs at the same time
We create functions for parsing several pdfs at the same time

In [6]:

def collect_pdf_filenames(file_path):
    # Get all PDF files recursively
    pdf_files = list(Path(file_path).glob("**/*.pdf"))

    return pdf_files

# Test the function
pdf_list = collect_pdf_filenames(DATA_PATH / 'Ausarbeitungen')
pdf_list

[PosixPath('../03_data/Ausarbeitungen/WD 10-013-23.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 1-019-24; WD 7-060-24.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 2-029-25.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 6-094-23.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 8-011-22.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 10-042-22.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 9-100-21.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 4-086-24.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 2-027-25_EN.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 6-052-24.pdf'),
 PosixPath('../03_data/Ausarbeitungen/EU 6-012-25.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 3-029-23.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 8-013-22.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 7-085-22; WD 5-124-22.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 5-009-25.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 9-068-23.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 7-051-24.pdf')]

In [7]:

def extract_metadata_from_folder(folder_path: Path):
    """Recursively extract metadata from all PDF files in a folder."""
    metadata_dict = {}

    pdf_list = collect_pdf_filenames(folder_path)
    for pdf_file in pdf_list:
            metadata_dict[pdf_file] = extract_pdf_metadata(pdf_file)

    print(f"{len(metadata_dict)}/{len(pdf_list)} documents processed")

    return metadata_dict, pdf_list

# Test function
metadata_dict = extract_metadata_from_folder(DATA_PATH / 'Ausarbeitungen')
metadata_dict

17/17 documents processed


({PosixPath('../03_data/Ausarbeitungen/WD 10-013-23.pdf'): {'Author': 'Wissenschaftliche Dienste des Deutschen Bundestages',
   'Keywords': '',
   'Producer': 'axesWord',
   'Creator': 'Microsoft® Word 2013 (15.0.5571)',
   'Subject': '',
   'Title': 'Spitzensportförderung in ausgewählten europäischen Ländern',
   'CreationDate': datetime.datetime(2023, 7, 25, 12, 3, 53, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))),
   'ModDate': datetime.datetime(2023, 7, 25, 12, 3, 53, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200)))},
  PosixPath('../03_data/Ausarbeitungen/WD 1-019-24; WD 7-060-24.pdf'): {'Subject': '',
   'Author': 'Wissenschaftliche Dienste des Deutschen Bundestages',
   'Keywords': '',
   'CreationDate': datetime.datetime(2024, 10, 17, 11, 20, 32, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))),
   'ModDate': datetime.datetime(2024, 10, 17, 11, 20, 32, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))),
   'Title': 'Sachstand - Die L

A class called <code>PDFMetadataExtractor</code> containing the previous logic has been added to <code>src.exp_a001.utils</code>.

### 6. Analyzing the data with pandas
We dump the data into a pandas dataframe for analyzing it.

In [8]:

metadata_dict, pdf_list = extract_metadata_from_folder(DATA_PATH / 'Extra')
df = pd.DataFrame.from_dict(
    {k: v for k, v in metadata_dict.items()},
    orient="index"
)
df


1949/1949 documents processed


,Subject,Author,Keywords,CreationDate,ModDate,Title,Creator,Producer,Comments,Dokumenttyp
../03_data/Extra/Aktueller Begriff Europa/EU-05-25.pdf,,Deutscher Bundestag - Unterabteilung Europa,,2025-07-15 09:13:56+02:00,2025-07-15 09:13:56+02:00,Zukunft der Sicherheits- und Verteidigungspoli...,NaN,NaN,NaN,NaN
../03_data/Extra/Aktueller Begriff Europa/AB Europa 02-22 Intranet und Internet.pdf,,NaN,,2022-07-14 09:44:53+02:00,2022-07-15 11:41:35+02:00,Aktueller Begriff Tschechiens Schwerpunkte für...,Microsoft® Word 2013,axesWord,NaN,NaN
../03_data/Extra/Aktueller Begriff Europa/EU-02-24.pdf,,Deutscher Bundestag - Unterabteilung Europa,,2024-08-21 15:07:38+02:00,2024-08-21 15:07:38+02:00,Aktueller Begriff Europa - Auswahl der Richter...,NaN,NaN,NaN,NaN
../03_data/Extra/Aktueller Begriff Europa/EU 06-25.pdf,,Deutscher Bundestag - Fachbereich Europa,,2025-07-04 13:15:27+02:00,2025-07-04 13:15:27+02:00,Schwerpunkte der dänischen EU-Ratspräsidentsch...,NaN,NaN,NaN,NaN
../03_data/Extra/Aktueller Begriff Europa/EU 01-24.pdf,,Deutscher Bundestag - Unterabteilung Europa,,2024-08-07 11:25:22+02:00,2024-08-07 11:25:22+02:00,Aktueller Begriff Europa - Die Prioritäten der...,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
../03_data/Extra/ab_2023/Ausarbeitungen/WD 9/2021/WD 9-045-21.pdf,NaN,NaN,NaN,2021-08-03 15:00:05+02:00,2021-08-03 15:00:05+02:00,NaN,Microsoft® Word 2013,Microsoft® Word 2013,NaN,NaN
../03_data/Extra/ab_2023/Ausarbeitungen/WD 9/2021/WD 9-106-21.pdf,NaN,NaN,NaN,2022-03-10 11:01:47+01:00,2022-03-10 11:01:47+01:00,NaN,Microsoft® Word 2013,Microsoft® Word 2013,NaN,NaN
../03_data/Extra/ab_2023/Ausarbeitungen/WD 9/2021/WD 9-096-21.pdf,NaN,NaN,NaN,2021-11-29 12:14:23+01:00,2021-11-29 12:14:23+01:00,NaN,Microsoft® Word 2013,Microsoft® Word 2013,NaN,NaN
../03_data/Extra/ab_2023/Ausarbeitungen/WD 10/2021/WD 10-027-21.pdf,NaN,NaN,NaN,2021-06-01 10:35:31+02:00,2021-06-07 08:54:41+02:00,NaN,Microsoft Word - WD 10-027-21 COVID-19 - Kultu...,Nuance PDF Create,NaN,NaN


Only 1915 from 1949 documents were processed

In [9]:
pdfs = [str(item) for item in pdf_list]
indexes = [str(item) for item in df.index]
diff_list = [x for x in pdfs if x not in indexes]
diff_list

['../03_data/Extra/ab_2023/Ausarbeitungen/WD 5/WD 5-148-07.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/WD 9/2021/WD 9-081-21; WD 3-164-21.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/PE 6/2023/EU 6-073-23.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/PE 6/2024/EU 6-031-24.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/PE 6/2024/EU 6-002-24.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/WD 4/2024/WD 4-075-24.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/WD 10/2021/WD 10-007-21.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/WD 10/2021/WD 10-019-21.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/WD 10/2021/WD 10-038-21.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/WD 10/2021/WD 10-033-21.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/WD 10/2021/WD 10-053-21.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/WD 10/2021/WD 10-051-21.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/WD 10/2021/WD 10-017-21.pdf',
 '../03_data/Extra/ab_2023/Ausarbeitungen/WD 10/2021/WD 10-024-21.pd